1 — Set paths + show environment

In [1]:
from pathlib import Path
import sys, os

THESIS_ROOT = Path("/home/psivakumar/thesis")
CODE_DIR    = THESIS_ROOT / "code"
DATA_DIR    = THESIS_ROOT / "data" / "iu_xray"

print("Python:", sys.version.split()[0])
print("THESIS_ROOT:", THESIS_ROOT, "| exists:", THESIS_ROOT.exists())
print("CODE_DIR   :", CODE_DIR,    "| exists:", CODE_DIR.exists())
print("DATA_DIR   :", DATA_DIR,    "| exists:", DATA_DIR.exists())

# show key files in dataset folder (should include CSVs + images folder)
if DATA_DIR.exists():
    print("\nDATA_DIR contents (top):")
    for p in sorted(DATA_DIR.iterdir())[:30]:
        print(" -", p.name)

Python: 3.10.19
THESIS_ROOT: /home/psivakumar/thesis | exists: True
CODE_DIR   : /home/psivakumar/thesis/code | exists: True
DATA_DIR   : /home/psivakumar/thesis/data/iu_xray | exists: True

DATA_DIR contents (top):
 - .ipynb_checkpoints
 - annotation.json
 - annotation_full.json
 - annotation_r2gencmn.json
 - annotation_r2gencmn_2view.json
 - eda_stats.json
 - images
 - indiana_projections.csv
 - indiana_reports.csv
 - iu_xray_pairs.csv
 - iu_xray_pairs_test.csv
 - iu_xray_pairs_train.csv
 - iu_xray_pairs_val.csv
 - runs
 - split_uids.json
 - splits


2 — Clone R2GenCMN

In [2]:
# Clone R2GenCMN repository if not already present
import os
from pathlib import Path

repo_dir = Path("/home/psivakumar/thesis/code/R2GenCMN")

if repo_dir.exists():
    print("R2GenCMN already exists at:", repo_dir)
else:
    !git clone https://github.com/cuhksz-nlp/R2GenCMN.git /home/psivakumar/thesis/code/R2GenCMN

# Verify contents
!ls -lah /home/psivakumar/thesis/code/R2GenCMN

R2GenCMN already exists at: /home/psivakumar/thesis/code/R2GenCMN
total 40K
drwxr-xr-x. 9 psivakumar psivakumar 4.0K Jan  9 14:01 .
drwxr-xr-x. 4 psivakumar psivakumar   48 Jan  9 13:38 ..
drwxr-xr-x. 3 psivakumar psivakumar   40 Jan  9 13:40 data
drwxr-xr-x. 8 psivakumar psivakumar  163 Jan  9 13:38 .git
drwxr-xr-x. 2 psivakumar psivakumar   32 Jan  9 14:01 .ipynb_checkpoints
-rw-r--r--. 1 psivakumar psivakumar  12K Jan  9 13:38 LICENSE
-rw-r--r--. 1 psivakumar psivakumar 7.5K Jan  9 13:38 main.py
drwxr-xr-x. 3 psivakumar psivakumar   42 Jan  9 13:54 models
drwxr-xr-x. 3 psivakumar psivakumar 4.0K Jan  9 13:54 modules
drwxr-xr-x. 3 psivakumar psivakumar   35 Jan  9 13:54 outputs
-rw-r--r--. 1 psivakumar psivakumar 1.6K Jan  9 13:38 README.md
drwxr-xr-x. 3 psivakumar psivakumar   35 Jan  9 13:54 records
-rw-r--r--. 1 psivakumar psivakumar  466 Jan  9 13:38 run_iu_xray.sh
-rw-r--r--. 1 psivakumar psivakumar  476 Jan  9 13:38 run_mimic_cxr.sh


3 — Create IU X-ray data folder + correct symlinks

In [3]:
from pathlib import Path
import os

THESIS_ROOT = Path("/home/psivakumar/thesis")
R2GEN_DIR   = THESIS_ROOT / "code" / "R2GenCMN"
R2GEN_IU    = R2GEN_DIR / "data" / "iu_xray"

DATA_DIR    = THESIS_ROOT / "data" / "iu_xray"
IMG_SRC     = DATA_DIR / "images" / "images_normalized"   # ✅ real images are here

# create folder
R2GEN_IU.mkdir(parents=True, exist_ok=True)

# create/overwrite symlink: R2GenCMN/data/iu_xray/images -> images_normalized
img_link = R2GEN_IU / "images"
if img_link.exists() or img_link.is_symlink():
    img_link.unlink()
img_link.symlink_to(IMG_SRC)

print("✅ Linked images:")
print("  ", img_link, "->", img_link.resolve())

# verify file count
png_count = sum(1 for _ in img_link.rglob("*.png"))
print("✅ PNG count:", png_count)

# list a few sample files
sample = list(img_link.glob("*.png"))[:5]
print("✅ Example files:", [p.name for p in sample])

✅ Linked images:
   /home/psivakumar/thesis/code/R2GenCMN/data/iu_xray/images -> /home/psivakumar/thesis/data/iu_xray/images/images_normalized
✅ PNG count: 7470
✅ Example files: ['1000_IM-0003-1001.dcm.png', '1000_IM-0003-2001.dcm.png', '1000_IM-0003-3001.dcm.png', '1001_IM-0004-1001.dcm.png', '1001_IM-0004-1002.dcm.png']


4 - Build annotation_r2gencmn.json from your CSV splits

In [4]:
import pandas as pd
import re, json
from pathlib import Path

THESIS_ROOT = Path("/home/psivakumar/thesis")
DATA_DIR    = THESIS_ROOT / "data" / "iu_xray"
R2GEN_IU    = THESIS_ROOT / "code" / "R2GenCMN" / "data" / "iu_xray"

TRAIN_CSV = DATA_DIR / "iu_xray_pairs_train.csv"
VAL_CSV   = DATA_DIR / "iu_xray_pairs_val.csv"
TEST_CSV  = DATA_DIR / "iu_xray_pairs_test.csv"

def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s)
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def build_split(csv_path: Path, split_name: str):
    df = pd.read_csv(csv_path)

    uid_col = pick_col(df, ["uid", "study_id", "id", "image_id"])
    img_col = pick_col(df, ["image_path", "image", "filename", "file", "path"])
    rep_col = pick_col(df, ["report_text", "report", "findings", "impression", "text"])

    print(f"\n== {split_name.upper()} ==")
    print("CSV:", csv_path.name)
    print("Columns:", list(df.columns))
    print("Using uid_col:", uid_col, "| img_col:", img_col, "| rep_col:", rep_col)

    if uid_col is None or img_col is None or rep_col is None:
        raise ValueError("❌ Could not detect required columns (uid/image/report).")

    df[uid_col] = df[uid_col].astype(str)
    df[img_col] = df[img_col].astype(str)
    df[rep_col] = df[rep_col].apply(normalize_text)

    entries = []
    grouped = df.groupby(uid_col, sort=False)

    for uid, g in grouped:
        imgs = list(dict.fromkeys(g[img_col].tolist()))  # unique, preserve order
        # In your CSV, image_path might contain full paths; keep only filename
        imgs = [Path(x).name for x in imgs]
        image_paths = [f"images/{x}" for x in imgs]

        report = ""
        for t in g[rep_col].tolist():
            if t:
                report = t
                break

        if not report:
            continue

        entries.append({
            "id": uid,
            "split": split_name,
            "image_path": image_paths,
            "report": report
        })

    print("Built entries:", len(entries))
    return entries

train_entries = build_split(TRAIN_CSV, "train")
val_entries   = build_split(VAL_CSV, "val")
test_entries  = build_split(TEST_CSV, "test")

all_entries = train_entries + val_entries + test_entries
out_json = DATA_DIR / "annotation_r2gencmn.json"

with open(out_json, "w") as f:
    json.dump(all_entries, f)

print("\n✅ Saved:", out_json)
print("✅ Total entries:", len(all_entries))
print("✅ Example entry:\n", json.dumps(all_entries[0], indent=2)[:900])


== TRAIN ==
CSV: iu_xray_pairs_train.csv
Columns: ['uid', 'filename', 'projection', 'image_path', 'report_text']
Using uid_col: uid | img_col: image_path | rep_col: report_text
Built entries: 3569

== VAL ==
CSV: iu_xray_pairs_val.csv
Columns: ['uid', 'filename', 'projection', 'image_path', 'report_text']
Using uid_col: uid | img_col: image_path | rep_col: report_text
Built entries: 685

== TEST ==
CSV: iu_xray_pairs_test.csv
Columns: ['uid', 'filename', 'projection', 'image_path', 'report_text']
Using uid_col: uid | img_col: image_path | rep_col: report_text
Built entries: 696

✅ Saved: /home/psivakumar/thesis/data/iu_xray/annotation_r2gencmn.json
✅ Total entries: 4950
✅ Example entry:
 {
  "id": "597",
  "split": "train",
  "image_path": [
    "images/597_IM-2189-2001.dcm.png"
  ],
  "report": "There are XXXX bilateral lower lobe opacities. No pleural effusion. No pneumothorax is identified. Heart size and mediastinal contour are within normal limits. There is lucency beneath the di

5 — Validate multi-view integrity + file existence

In [5]:
import json
from collections import Counter
from pathlib import Path

THESIS_ROOT = Path("/home/psivakumar/thesis")
DATA_DIR    = THESIS_ROOT / "data" / "iu_xray"
R2GEN_IU    = THESIS_ROOT / "code" / "R2GenCMN" / "data" / "iu_xray"

ann_path = DATA_DIR / "annotation_r2gencmn.json"
data = json.load(open(ann_path))

print("Loaded entries:", len(data))

# Count number of images per UID
img_counts = Counter(len(x["image_path"]) for x in data)
print("\nImage count distribution (how many UIDs have N images):")
for k in sorted(img_counts):
    print(f"  {k} images: {img_counts[k]}")

# Check existence of files
missing = []
for x in data[:200]:  # quick first 200
    for p in x["image_path"]:
        full = R2GEN_IU / p  # p is "images/....png"
        if not full.exists():
            missing.append((x["id"], p))
            break

print("\nQuick existence check on first 200 entries:")
print("  Missing count:", len(missing))
if missing[:5]:
    print("  Examples:", missing[:5])

# Show examples of 1-image and 2-image UIDs
one_img = [x for x in data if len(x["image_path"]) == 1][:3]
two_img = [x for x in data if len(x["image_path"]) == 2][:3]

print("\nExamples (1 image):")
for x in one_img:
    print(" UID:", x["id"], "|", x["image_path"])

print("\nExamples (2 images):")
for x in two_img:
    print(" UID:", x["id"], "|", x["image_path"])

Loaded entries: 4950

Image count distribution (how many UIDs have N images):
  1 images: 2617
  2 images: 2228
  3 images: 96
  4 images: 8
  5 images: 1

Quick existence check on first 200 entries:
  Missing count: 0

Examples (1 image):
 UID: 597 | ['images/597_IM-2189-2001.dcm.png']
 UID: 1052 | ['images/1052_IM-0040-1001.dcm.png']
 UID: 3412 | ['images/3412_IM-1650-1001-0002.dcm.png']

Examples (2 images):
 UID: 3931 | ['images/3931_IM-2003-3001.dcm.png', 'images/3931_IM-2003-1001.dcm.png']
 UID: 3813 | ['images/3813_IM-1922-1001.dcm.png', 'images/3813_IM-1922-2001.dcm.png']
 UID: 1713 | ['images/1713_IM-0471-4004.dcm.png', 'images/1713_IM-0471-1001.dcm.png']


6 - Build 2-view only annotation (frontal + lateral) using projection

In [6]:
import pandas as pd
import re, json
from pathlib import Path
from collections import Counter

THESIS_ROOT = Path("/home/psivakumar/thesis")
DATA_DIR    = THESIS_ROOT / "data" / "iu_xray"

TRAIN_CSV = DATA_DIR / "iu_xray_pairs_train.csv"
VAL_CSV   = DATA_DIR / "iu_xray_pairs_val.csv"
TEST_CSV  = DATA_DIR / "iu_xray_pairs_test.csv"

def normalize_text(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def proj_rank(proj: str) -> int:
    """
    Put frontal first, lateral second.
    Robust to variations: 'Frontal', 'Lateral', 'PA', 'AP'.
    """
    if proj is None: 
        return 99
    p = str(proj).lower()
    if "frontal" in p or p in ("pa", "ap") or "postero" in p or "antero" in p:
        return 0
    if "lateral" in p:
        return 1
    return 50

def build_2view(csv_path: Path, split_name: str):
    df = pd.read_csv(csv_path)

    # known columns from your dataset
    uid_col  = "uid"
    img_col  = "image_path"
    rep_col  = "report_text"
    prj_col  = "projection"

    df[uid_col] = df[uid_col].astype(str)
    df[img_col] = df[img_col].astype(str).apply(lambda x: Path(x).name)
    df[rep_col] = df[rep_col].apply(normalize_text)

    entries = []
    grouped = df.groupby(uid_col, sort=False)

    drop_counts = Counter()

    for uid, g in grouped:
        g = g.drop_duplicates(subset=[img_col])

        if len(g) != 2:
            drop_counts[len(g)] += 1
            continue

        g = g.assign(_rank=g[prj_col].apply(proj_rank)).sort_values("_rank")

        imgs = g[img_col].tolist()
        report = g[rep_col].iloc[0]

        if not report:
            continue

        entries.append({
            "id": uid,
            "split": split_name,
            "image_path": [f"images/{imgs[0]}", f"images/{imgs[1]}"],
            "report": report
        })

    print(f"\n== {split_name.upper()} 2-VIEW ONLY ==")
    print("Built entries:", len(entries))
    print("Dropped UID groups by image-count:", dict(drop_counts))
    return entries

train_2v = build_2view(TRAIN_CSV, "train")
val_2v   = build_2view(VAL_CSV, "val")
test_2v  = build_2view(TEST_CSV, "test")

all_2v = train_2v + val_2v + test_2v
out_2v = DATA_DIR / "annotation_r2gencmn_2view.json"

with open(out_2v, "w") as f:
    json.dump(all_2v, f)

print("\n✅ Saved:", out_2v)
print("✅ Total 2-view entries:", len(all_2v))
print("✅ Example:\n", json.dumps(all_2v[0], indent=2)[:900])


== TRAIN 2-VIEW ONLY ==
Built entries: 2131
Dropped UID groups by image-count: {1: 1334, 3: 95, 5: 1, 4: 8}

== VAL 2-VIEW ONLY ==
Built entries: 53
Dropped UID groups by image-count: {1: 631, 3: 1}

== TEST 2-VIEW ONLY ==
Built entries: 44
Dropped UID groups by image-count: {1: 652}

✅ Saved: /home/psivakumar/thesis/data/iu_xray/annotation_r2gencmn_2view.json
✅ Total 2-view entries: 2228
✅ Example:
 {
  "id": "3931",
  "split": "train",
  "image_path": [
    "images/3931_IM-2003-3001.dcm.png",
    "images/3931_IM-2003-1001.dcm.png"
  ],
  "report": "The heart size is normal. The cardiomediastinal silhouette is stable in appearance. The lungs are clear without focal airspace opacity, pneumothorax, or pleural effusion. The XXXX are normal in appearance. IMPRESSION: No acute cardiopulmonary finding."
}


7 — Final validation + link into R2GenCMN

In [7]:
import json
from pathlib import Path
from collections import Counter

THESIS_ROOT = Path("/home/psivakumar/thesis")
DATA_DIR    = THESIS_ROOT / "data" / "iu_xray"
R2GEN_IU    = THESIS_ROOT / "code" / "R2GenCMN" / "data" / "iu_xray"

ann_2v = DATA_DIR / "annotation_r2gencmn_2view.json"
data = json.load(open(ann_2v))

print("Loaded entries:", len(data))

# 1) verify exactly 2 images per UID
cnt = Counter(len(x["image_path"]) for x in data)
print("Image count distribution:", dict(cnt))

# 2) verify files exist
missing = []
for x in data[:300]:  # quick check
    for p in x["image_path"]:
        full = R2GEN_IU / p
        if not full.exists():
            missing.append((x["id"], p))
            break

print("Missing files in first 300:", len(missing))
if missing:
    print("Examples:", missing[:3])

# 3) show a few examples
print("\nSample entries:")
for x in data[:3]:
    print("UID:", x["id"])
    print(" Images:", x["image_path"])
    print(" Report length:", len(x["report"]))
    print("---")

# 4) link annotation.json for R2GenCMN
target = R2GEN_IU / "annotation.json"
if target.exists() or target.is_symlink():
    target.unlink()
target.symlink_to(ann_2v)

print("\n✅ Linked annotation.json ->", target.resolve())

Loaded entries: 2228
Image count distribution: {2: 2228}
Missing files in first 300: 0

Sample entries:
UID: 3931
 Images: ['images/3931_IM-2003-3001.dcm.png', 'images/3931_IM-2003-1001.dcm.png']
 Report length: 251
---
UID: 3813
 Images: ['images/3813_IM-1922-1001.dcm.png', 'images/3813_IM-1922-2001.dcm.png']
 Report length: 220
---
UID: 1713
 Images: ['images/1713_IM-0471-4004.dcm.png', 'images/1713_IM-0471-1001.dcm.png']
 Report length: 198
---

✅ Linked annotation.json -> /home/psivakumar/thesis/data/iu_xray/annotation_r2gencmn_2view.json


8 — Create annotation_tiny_test.json (small subset for 1-epoch smoke run)

In [8]:
import json, random
from pathlib import Path

THESIS_ROOT = Path("/home/psivakumar/thesis")
DATA_DIR    = THESIS_ROOT / "data" / "iu_xray"

ann_path = DATA_DIR / "annotation_r2gencmn_2view.json"
data = json.load(open(ann_path))

# split lists
train = [x for x in data if x["split"] == "train"]
val   = [x for x in data if x["split"] == "val"]
test  = [x for x in data if x["split"] == "test"]

print("Full sizes:", {"train": len(train), "val": len(val), "test": len(test)})

random.seed(42)
tiny = (
    random.sample(train, min(10, len(train))) +
    random.sample(val,   min(5,  len(val))) +
    random.sample(test,  min(5,  len(test)))
)

out_tiny = (THESIS_ROOT / "code" / "R2GenCMN" / "data" / "iu_xray" / "annotation_tiny_test.json")
with open(out_tiny, "w") as f:
    json.dump(tiny, f)

print("✅ Saved tiny annotation:", out_tiny)
print("Tiny sizes:", {
    "train": sum(1 for x in tiny if x["split"]=="train"),
    "val":   sum(1 for x in tiny if x["split"]=="val"),
    "test":  sum(1 for x in tiny if x["split"]=="test"),
})
print("✅ Example:\n", json.dumps(tiny[0], indent=2)[:700])

Full sizes: {'train': 2131, 'val': 53, 'test': 44}
✅ Saved tiny annotation: /home/psivakumar/thesis/code/R2GenCMN/data/iu_xray/annotation_tiny_test.json
Tiny sizes: {'train': 10, 'val': 5, 'test': 5}
✅ Example:
 {
  "id": "2028",
  "split": "train",
  "image_path": [
    "images/2028_IM-0673-1001.dcm.png",
    "images/2028_IM-0673-1002.dcm.png"
  ],
  "report": "Heart size is normal. No focal airspace disease. No pneumothorax or effusion. IMPRESSION: No acute cardiopulmonary finding."
}


9 — Environment + CUDA/GPU check + key imports

In [9]:
import os, sys
from pathlib import Path

print("Python:", sys.version.split()[0])

# Torch / CUDA check
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda device:", torch.cuda.get_device_name(0))
        print("cuda capability:", torch.cuda.get_device_capability(0))
    else:
        print("⚠️ CUDA not available - training will run on CPU unless you switch to a GPU node.")
except Exception as e:
    print("❌ torch import failed:", repr(e))

# Quick check of GPU visibility via nvidia-smi (shell)
print("\n--- nvidia-smi ---")
os.system("nvidia-smi || true")

# Repo import sanity
print("\n--- R2GenCMN repo path ---")
repo = Path("/home/psivakumar/thesis/code/R2GenCMN")
print("Repo exists:", repo.exists(), "| contains main.py:", (repo/"main.py").exists())

Python: 3.10.19
torch: 2.5.1+cu121
cuda available: True
cuda device: NVIDIA L40S
cuda capability: (8, 9)

--- nvidia-smi ---
Failed to initialize NVML: Driver/library version mismatch
NVML library version: 590.48

--- R2GenCMN repo path ---
Repo exists: True | contains main.py: True


/home/psivakumar/.conda/envs/ml/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


10 — Smoke training (1 epoch) using tiny annotation

In [10]:
# Install dependencies required for evaluation metrics
# This installs COCO caption evaluation code

!pip install git+https://github.com/salaniz/pycocoevalcap

  Cloning https://github.com/salaniz/pycocoevalcap to /tmp/pip-req-build-rt3xydyw
  Running command git clone --filter=blob:none --quiet https://github.com/salaniz/pycocoevalcap /tmp/pip-req-build-rt3xydyw
  Resolved https://github.com/salaniz/pycocoevalcap to commit a24f74c408c918f1f4ec34e9514bc8a76ce41ffd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
# HARD RESET metrics.py (clean + safe)
from pathlib import Path

metrics_path = Path("/home/psivakumar/thesis/code/R2GenCMN/modules/metrics.py")

minimal_metrics = '''
"""
Minimal metrics stub.

NOTE:
COCO-style metrics (BLEU/METEOR/ROUGE/CIDEr) are disabled during training
due to dependency issues on HPC environments.
Evaluation is performed separately in an offline notebook.
"""

def compute_scores(gts, res):
    """
    Args:
        gts: dict of ground-truth reports
        res: dict of generated reports
    Returns:
        empty dict (metrics disabled during training)
    """
    return {}
'''

metrics_path.write_text(minimal_metrics.strip())

print("✅ metrics.py replaced with minimal stub")
print("File content:\n")
print(metrics_path.read_text())

✅ metrics.py replaced with minimal stub
File content:

"""
Minimal metrics stub.

NOTE:
COCO-style metrics (BLEU/METEOR/ROUGE/CIDEr) are disabled during training
due to dependency issues on HPC environments.
Evaluation is performed separately in an offline notebook.
"""

def compute_scores(gts, res):
    """
    Args:
        gts: dict of ground-truth reports
        res: dict of generated reports
    Returns:
        empty dict (metrics disabled during training)
    """
    return {}


In [13]:
# 13 — Convert annotation to R2GenCMN format (dict with train/val/test)
import json
from pathlib import Path

THESIS_ROOT = Path("/home/psivakumar/thesis")
DATA_DIR    = THESIS_ROOT / "data" / "iu_xray"
R2GEN_IU    = THESIS_ROOT / "code" / "R2GenCMN" / "data" / "iu_xray"

src_path = DATA_DIR / "annotation_r2gencmn_2view.json"  # our 2-view list with split field
data = json.load(open(src_path))

# Convert to dict format expected by R2GenCMN
out = {"train": [], "val": [], "test": []}
for x in data:
    split = x["split"]
    out[split].append({
        "id": x["id"],
        "image_path": x["image_path"],
        "report": x["report"]
    })

out_path = DATA_DIR / "annotation_r2gencmn_2view_r2gencmnformat.json"
with open(out_path, "w") as f:
    json.dump(out, f)

print("✅ Saved:", out_path)
print("Sizes:", {k: len(v) for k, v in out.items()})
print("Example train entry:\n", json.dumps(out["train"][0], indent=2)[:600])

# Link annotation.json used by R2GenCMN to this new file
link_path = R2GEN_IU / "annotation.json"
if link_path.exists() or link_path.is_symlink():
    link_path.unlink()
link_path.symlink_to(out_path)

print("\n✅ Linked R2GenCMN annotation.json ->", link_path.resolve())

✅ Saved: /home/psivakumar/thesis/data/iu_xray/annotation_r2gencmn_2view_r2gencmnformat.json
Sizes: {'train': 2131, 'val': 53, 'test': 44}
Example train entry:
 {
  "id": "3931",
  "image_path": [
    "images/3931_IM-2003-3001.dcm.png",
    "images/3931_IM-2003-1001.dcm.png"
  ],
  "report": "The heart size is normal. The cardiomediastinal silhouette is stable in appearance. The lungs are clear without focal airspace opacity, pneumothorax, or pleural effusion. The XXXX are normal in appearance. IMPRESSION: No acute cardiopulmonary finding."
}

✅ Linked R2GenCMN annotation.json -> /home/psivakumar/thesis/data/iu_xray/annotation_r2gencmn_2view_r2gencmnformat.json


In [15]:
# 14 — Convert tiny annotation to the same format (tiny dict)
import json
from pathlib import Path

THESIS_ROOT = Path("/home/psivakumar/thesis")
R2GEN_IU    = THESIS_ROOT / "code" / "R2GenCMN" / "data" / "iu_xray"

tiny_src = R2GEN_IU / "annotation_tiny_test.json"
tiny = json.load(open(tiny_src))

tiny_out = {"train": [], "val": [], "test": []}
for x in tiny:
    tiny_out[x["split"]].append({
        "id": x["id"],
        "image_path": x["image_path"],
        "report": x["report"]
    })

tiny_out_path = R2GEN_IU / "annotation_tiny_test_r2gencmnformat.json"
with open(tiny_out_path, "w") as f:
    json.dump(tiny_out, f)

print("✅ Saved tiny dict:", tiny_out_path)
print("Tiny sizes:", {k: len(v) for k, v in tiny_out.items()})

✅ Saved tiny dict: /home/psivakumar/thesis/code/R2GenCMN/data/iu_xray/annotation_tiny_test_r2gencmnformat.json
Tiny sizes: {'train': 10, 'val': 5, 'test': 5}


In [22]:
import os
from pathlib import Path

THESIS_ROOT = Path("/home/psivakumar/thesis")
R2GEN_DIR   = THESIS_ROOT / "code" / "R2GenCMN"

# ✅ Use the dict-format tiny annotation (train/val/test keys)
ann_tiny = R2GEN_DIR / "data" / "iu_xray" / "annotation_tiny_test_r2gencmnformat.json"

# Image dir is the IU folder inside R2GenCMN (contains "images/" symlink)
img_dir  = R2GEN_DIR / "data" / "iu_xray"

save_dir   = R2GEN_DIR / "outputs" / "iu_xray_r2gencmn_tiny"
record_dir = R2GEN_DIR / "records" / "iu_xray_r2gencmn_tiny"

# Safety checks
assert ann_tiny.exists(), f"❌ Tiny annotation not found: {ann_tiny}"
assert (img_dir / "images").exists(), f"❌ images link not found: {img_dir/'images'}"

# Make sure output dirs exist
save_dir.mkdir(parents=True, exist_ok=True)
record_dir.mkdir(parents=True, exist_ok=True)

cmd = f"""
cd {R2GEN_DIR} && \
python main.py \
  --dataset_name iu_xray \
  --image_dir {img_dir} \
  --ann_path {ann_tiny} \
  --save_dir {save_dir} \
  --record_dir {record_dir} \
  --epochs 1 \
  --batch_size 4
"""

print("Running command:\n", cmd)
ret = os.system(cmd)
print("Return code:", ret)

Running command:
 
cd /home/psivakumar/thesis/code/R2GenCMN && python main.py   --dataset_name iu_xray   --image_dir /home/psivakumar/thesis/code/R2GenCMN/data/iu_xray   --ann_path /home/psivakumar/thesis/code/R2GenCMN/data/iu_xray/annotation_tiny_test_r2gencmnformat.json   --save_dir /home/psivakumar/thesis/code/R2GenCMN/outputs/iu_xray_r2gencmn_tiny   --record_dir /home/psivakumar/thesis/code/R2GenCMN/records/iu_xray_r2gencmn_tiny   --epochs 1   --batch_size 4



/home/psivakumar/envs/ml/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/psivakumar/envs/ml/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/psivakumar/envs/ml/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
01/09/2026 14:19:35 - INFO - modules.trainer -   [1/1] Start to train in the training set.
01/09/2026 14:19:37 - INFO - modules.trainer -   [1/1] Step: 0/3, Training Loss: 3.50368.
01/09/20

Return code: 0


In [18]:
from pathlib import Path
import re

main_path = Path("/home/psivakumar/thesis/code/R2GenCMN/main.py")
src = main_path.read_text()

print("✅ main.py found:", main_path.exists())

# Show lines mentioning mnt_metric / mnt_mode
lines = src.splitlines()
for i, ln in enumerate(lines):
    if "mnt_metric" in ln or "mnt_mode" in ln:
        print(f"{i+1:04d}: {ln}")

✅ main.py found: True


In [21]:
from pathlib import Path
import re

trainer_path = Path("/home/psivakumar/thesis/code/R2GenCMN/modules/trainer.py")
txt = trainer_path.read_text()

# Locate the _record_best function
m = re.search(r"def _record_best\(self, log\):([\s\S]*?)def _print_best", txt)
assert m, "❌ Could not find _record_best() block to patch."

block = m.group(0)

# If already patched, don't double patch
if "SAFE_GUARD_RECORD_BEST" in block:
    print("✅ _record_best already patched. Skipping.")
else:
    # Insert guard right after function definition line
    patched_block = re.sub(
        r"def _record_best\(self, log\):\n",
        "def _record_best(self, log):\n"
        "        # SAFE_GUARD_RECORD_BEST\n"
        "        # If metrics are disabled/unavailable, skip best-recording safely.\n"
        "        if getattr(self, 'mnt_mode', 'off') == 'off':\n"
        "            return\n"
        "        if self.mnt_metric not in log:\n"
        "            return\n"
        "        if self.mnt_metric_test not in log:\n"
        "            return\n",
        block,
        count=1
    )

    # Replace old block with patched block
    txt = txt.replace(block, patched_block)
    trainer_path.write_text(txt)
    print("✅ Patched _record_best() with safe guards.")

# Print the patched function header for proof
new_txt = trainer_path.read_text()
start = new_txt.find("def _record_best")
end = new_txt.find("def _print_best", start)
print("\n--- Patched _record_best() preview ---")
print(new_txt[start:end].splitlines()[:25])

✅ Patched _record_best() with safe guards.

--- Patched _record_best() preview ---
['def _record_best(self, log):', '        # SAFE_GUARD_RECORD_BEST', '        # If metrics are disabled/unavailable, skip best-recording safely.', "        if getattr(self, 'mnt_mode', 'off') == 'off':", '            return', '        if self.mnt_metric not in log:', '            return', '        if self.mnt_metric_test not in log:', '            return', "        improved_val = (self.mnt_mode == 'min' and log[self.mnt_metric] <= self.best_recorder['val'][", '            self.mnt_metric]) or \\', "                       (self.mnt_mode == 'max' and log[self.mnt_metric] >= self.best_recorder['val'][self.mnt_metric])", '        if improved_val:', "            self.best_recorder['val'].update(log)", '', "        improved_test = (self.mnt_mode == 'min' and log[self.mnt_metric_test] <= self.best_recorder['test'][", '            self.mnt_metric_test]) or \\', "                        (self.mnt_mode == 'max' an

15 - Verify checkpoint + logs were created

In [23]:
from pathlib import Path

R2GEN_DIR = Path("/home/psivakumar/thesis/code/R2GenCMN")
save_dir   = R2GEN_DIR / "outputs" / "iu_xray_r2gencmn_tiny"
record_dir = R2GEN_DIR / "records" / "iu_xray_r2gencmn_tiny"

print("Save dir:", save_dir)
print("Record dir:", record_dir)

print("\n--- outputs ---")
for p in sorted(save_dir.glob("*")):
    print(p.name, "-", p.stat().st_size, "bytes")

print("\n--- records ---")
for p in sorted(record_dir.rglob("*"))[:50]:
    if p.is_file():
        print(p.relative_to(record_dir), "-", p.stat().st_size, "bytes")

Save dir: /home/psivakumar/thesis/code/R2GenCMN/outputs/iu_xray_r2gencmn_tiny
Record dir: /home/psivakumar/thesis/code/R2GenCMN/records/iu_xray_r2gencmn_tiny

--- outputs ---
current_checkpoint.pth - 954733386 bytes

--- records ---


18 — Full training run (baseline)

In [24]:
import os
from pathlib import Path

THESIS_ROOT = Path("/home/psivakumar/thesis")
R2GEN_DIR   = THESIS_ROOT / "code" / "R2GenCMN"

ann_full = THESIS_ROOT / "data" / "iu_xray" / "annotation_r2gencmn_2view_r2gencmnformat.json"
img_dir  = R2GEN_DIR / "data" / "iu_xray"

save_dir   = R2GEN_DIR / "outputs" / "iu_xray_r2gencmn_baseline_5ep"
record_dir = R2GEN_DIR / "records" / "iu_xray_r2gencmn_baseline_5ep"

assert ann_full.exists(), f"❌ Missing annotation: {ann_full}"
assert (img_dir / "images").exists(), f"❌ Missing images link: {img_dir/'images'}"

save_dir.mkdir(parents=True, exist_ok=True)
record_dir.mkdir(parents=True, exist_ok=True)

cmd = f"""
cd {R2GEN_DIR} && \
python main.py \
  --dataset_name iu_xray \
  --image_dir {img_dir} \
  --ann_path {ann_full} \
  --save_dir {save_dir} \
  --record_dir {record_dir} \
  --epochs 5 \
  --batch_size 8
"""

print("Running:\n", cmd)
ret = os.system(cmd)
print("Return code:", ret)

Running:
 
cd /home/psivakumar/thesis/code/R2GenCMN && python main.py   --dataset_name iu_xray   --image_dir /home/psivakumar/thesis/code/R2GenCMN/data/iu_xray   --ann_path /home/psivakumar/thesis/data/iu_xray/annotation_r2gencmn_2view_r2gencmnformat.json   --save_dir /home/psivakumar/thesis/code/R2GenCMN/outputs/iu_xray_r2gencmn_baseline_5ep   --record_dir /home/psivakumar/thesis/code/R2GenCMN/records/iu_xray_r2gencmn_baseline_5ep   --epochs 5   --batch_size 8



/home/psivakumar/envs/ml/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/psivakumar/envs/ml/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/psivakumar/envs/ml/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
01/09/2026 14:27:59 - INFO - modules.trainer -   [1/5] Start to train in the training set.
01/09/2026 14:28:02 - INFO - modules.trainer -   [1/5] Step: 0/267, Training Loss: 6.96948.
01/09/

Return code: 0


19 — Find how R2GenCMN does inference (search in repo)

In [26]:
import os, re
from pathlib import Path

R2GEN_DIR = Path("/home/psivakumar/thesis/code/R2GenCMN")

targets = [
    "generate", "inference", "beam", "caption", "greedy", "decode", "sample", "test"
]

hits = []
for py in R2GEN_DIR.rglob("*.py"):
    txt = py.read_text(errors="ignore")
    for t in targets:
        if re.search(rf"\b{t}\b", txt):
            hits.append(str(py.relative_to(R2GEN_DIR)))
            break

print("Files likely related to inference:")
for h in sorted(set(hits)):
    print(" -", h)

Files likely related to inference:
 - .ipynb_checkpoints/main-checkpoint.py
 - main.py
 - models/models.py
 - modules/att_model.py
 - modules/base_cmn.py
 - modules/caption_model.py
 - modules/datasets.py
 - modules/tokenizers.py
 - modules/trainer.py


20 — Locate where gts and res are built (with line numbers)

In [27]:
from pathlib import Path

trainer_path = Path("/home/psivakumar/thesis/code/R2GenCMN/modules/trainer.py")
lines = trainer_path.read_text().splitlines()

keywords = ["gts", "res", "compute_scores", "decode", "greedy", "beam", "generate"]
print("✅ trainer.py length:", len(lines), "lines")

print("\n--- Lines containing key inference/metric variables ---")
for i, ln in enumerate(lines, start=1):
    if any(k in ln for k in keywords):
        print(f"{i:04d}: {ln}")

✅ trainer.py length: 238 lines

--- Lines containing key inference/metric variables ---
0048:         if args.resume is not None:
0049:             self._resume_checkpoint(args.resume)
0058:             result = self._train_epoch(epoch)
0062:             log.update(result)
0121:         self.logger.info('Best results (w.r.t {}) in validation set:'.format(self.args.monitor_metric))
0125:         self.logger.info('Best results (w.r.t {}) in test set:'.format(self.args.monitor_metric))
0159:     def _resume_checkpoint(self, resume_path):
0160:         resume_path = str(resume_path)
0161:         self.logger.info("Loading checkpoint: {} ...".format(resume_path))
0162:         checkpoint = torch.load(resume_path)
0204:             val_gts, val_res = [], []
0210:                 reports = self.model.tokenizer.decode_batch(output.cpu().numpy())
0211:                 ground_truths = self.model.tokenizer.decode_batch(reports_ids[:, 1:].cpu().numpy())
0212:                 val_res.extend(reports

21 — Patch trainer.py to dump predictions JSON (val + test)

In [28]:
from pathlib import Path
import re

trainer_path = Path("/home/psivakumar/thesis/code/R2GenCMN/modules/trainer.py")
text = trainer_path.read_text()

# Avoid double patch
if "SAVE_PREDS_JSON" in text:
    print("✅ trainer.py already patched for saving preds.")
else:
    # Insert helper function near top (after imports). We'll place after first occurrence of "import" block end.
    insert_marker = "class Trainer"
    helper = """
# SAVE_PREDS_JSON
import json

def _safe_get_uids(batch, fallback_start=0):
    \"\"\"Try common keys for UID; fallback to running indices.\"\"\"
    for k in ["ids", "id", "uid", "study_id", "image_id"]:
        if k in batch:
            v = batch[k]
            try:
                # torch tensor or list
                if hasattr(v, "cpu"):
                    v = v.cpu().numpy().tolist()
                return [str(x) for x in v]
            except Exception:
                return [str(x) for x in v]
    # fallback
    return [str(fallback_start + i) for i in range(len(batch["images"]))] if "images" in batch else [str(fallback_start + i) for i in range(len(batch["reports_ids"]))]

def _write_preds_json(path, uids, gts, preds):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    for uid, gt, pr in zip(uids, gts, preds):
        rows.append({"uid": str(uid), "gt": gt, "pred": pr})
    path.write_text(json.dumps(rows, indent=2))
"""
    text = text.replace(insert_marker, helper + "\n\n" + insert_marker, 1)

    # Now patch val loop: create val_uids list and extend it per batch, then write file after loop
    # We'll inject val_uids near val_gts/val_res init.
    text = text.replace(
        "val_gts, val_res = [], []",
        "val_gts, val_res, val_uids = [], [], []"
    )

    # Inside val loop, after ground_truths computed, add uids extraction
    # We insert after line containing "ground_truths ="
    text = re.sub(
        r"(ground_truths\s*=\s*self\.model\.tokenizer\.decode_batch\(reports_ids\[:, 1:\]\.cpu\(\)\.numpy\(\)\)\n)",
        r"\1                uids = _safe_get_uids(batch, fallback_start=len(val_uids))\n                val_uids.extend(uids)\n",
        text
    )

    # After val loop ends, before val_met computation, dump preds json
    text = text.replace(
        "val_met = self.metric_ftns({i: [gt] for i, gt in enumerate(val_gts)},",
        "                _write_preds_json(Path(self.args.record_dir)/'preds_val.json', val_uids, val_gts, val_res)\n"
        "            val_met = self.metric_ftns({i: [gt] for i, gt in enumerate(val_gts)},"
    )

    # Patch test similarly
    text = text.replace(
        "test_gts, test_res = [], []",
        "test_gts, test_res, test_uids = [], [], []"
    )

    text = re.sub(
        r"(ground_truths\s*=\s*self\.model\.tokenizer\.decode_batch\(reports_ids\[:, 1:\]\.cpu\(\)\.numpy\(\)\)\n)",
        r"\1                uids = _safe_get_uids(batch, fallback_start=len(test_uids))\n                test_uids.extend(uids)\n",
        text,
        count=1  # ⚠️ only patch the next occurrence (test loop) once more below
    )
    # The above might hit val again depending on ordering, so do a second pass specifically after 'test_res.extend'
    # safer: insert write after test loop, before test_met
    text = text.replace(
        "test_met = self.metric_ftns({i: [gt] for i, gt in enumerate(test_gts)},",
        "                _write_preds_json(Path(self.args.record_dir)/'preds_test.json', test_uids, test_gts, test_res)\n"
        "            test_met = self.metric_ftns({i: [gt] for i, gt in enumerate(test_gts)},"
    )

    trainer_path.write_text(text)
    print("✅ Patched trainer.py to save preds_val.json and preds_test.json")

print("✅ Done.")

✅ Patched trainer.py to save preds_val.json and preds_test.json
✅ Done.


22 — Run a “resume only” pass to dump predictions (no retraining)

In [29]:
import os
from pathlib import Path

THESIS_ROOT = Path("/home/psivakumar/thesis")
R2GEN_DIR   = THESIS_ROOT / "code" / "R2GenCMN"

ann_full = THESIS_ROOT / "data" / "iu_xray" / "annotation_r2gencmn_2view_r2gencmnformat.json"
img_dir  = R2GEN_DIR / "data" / "iu_xray"

save_dir   = R2GEN_DIR / "outputs" / "iu_xray_r2gencmn_baseline_5ep"
record_dir = R2GEN_DIR / "records" / "iu_xray_r2gencmn_baseline_5ep"

ckpt = save_dir / "current_checkpoint.pth"
assert ckpt.exists(), f"Missing checkpoint: {ckpt}"

cmd = f"""
cd {R2GEN_DIR} && \
python main.py \
  --dataset_name iu_xray \
  --image_dir {img_dir} \
  --ann_path {ann_full} \
  --save_dir {save_dir} \
  --record_dir {record_dir} \
  --resume {ckpt} \
  --epochs 1 \
  --batch_size 8
"""
print(cmd)
os.system(cmd)

print("\nCheck dumped files:")
os.system(f"ls -lah {record_dir} | head -n 50")


cd /home/psivakumar/thesis/code/R2GenCMN && python main.py   --dataset_name iu_xray   --image_dir /home/psivakumar/thesis/code/R2GenCMN/data/iu_xray   --ann_path /home/psivakumar/thesis/data/iu_xray/annotation_r2gencmn_2view_r2gencmnformat.json   --save_dir /home/psivakumar/thesis/code/R2GenCMN/outputs/iu_xray_r2gencmn_baseline_5ep   --record_dir /home/psivakumar/thesis/code/R2GenCMN/records/iu_xray_r2gencmn_baseline_5ep   --resume /home/psivakumar/thesis/code/R2GenCMN/outputs/iu_xray_r2gencmn_baseline_5ep/current_checkpoint.pth   --epochs 1   --batch_size 8



Traceback (most recent call last):
  File "/home/psivakumar/thesis/code/R2GenCMN/main.py", line 12, in <module>
    from modules.trainer import Trainer
  File "/home/psivakumar/thesis/code/R2GenCMN/modules/trainer.py", line 247
    _write_preds_json(Path(self.args.record_dir)/'preds_val.json', val_uids, val_gts, val_res)
IndentationError: unexpected indent



Check dumped files:
total 0
drwxr-xr-x. 2 psivakumar psivakumar  6 Jan  9 14:27 .
drwxr-xr-x. 4 psivakumar psivakumar 72 Jan  9 14:27 ..


0